In [3]:
%pip install -e .

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///Users/li7186lu/Desktop/li7186lu/Work%20Folders/Data/ApoE_Project/APOE_proteomics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sumAPOE (pyproject.toml) ... done
  Created wheel for sumAPOE: filename=sumapoe-0.1.0-0.editable-py3-none-any.whl size=2151 sha256=0cc86632ebdf1ac27ec02aec4ee9e87c0a46756434659ab6382281433bdbca9f
  Stored in directory: /private/var/folders/81/ddj0xhs55bg3d4mpqcms0bjw0000gp/T/pip-ephem-wheel-cache-u3__tr4h/wheels/eb/f0/dc/377d66ecf7f1520a8b42cc92407cc1317c18829b2be0edfc53
Successfully built sumAPOE
  Attempting uninstall: sumAPOE
    Found existing installation: sumAPOE 0.1.0
    Uninstalling sumAPOE-0.1.0:
      Successfully uninstalled sumAPOE-0.1.0
Note: you may need to restart the 

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import statsmodels.api as sm
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter
from scipy.stats import zscore
from sumAPOE import ResultDataLoader 

In [5]:
apoe_identifier = 'APOE4'

In [ ]:
initial = pd.read_csv('analysis/clean_withe2e4.csv', index_col=0)
initial = initial[initial['contributor_code'] != 'C']

if apoe_identifier == 'APOE4':
    df = initial[initial['APOE'].isin([33, 34, 43, 44])] # APOE4
    datafolder = 'results/GNPC/e4vse3e3'
    outpath = 'results/GNPC/e4vse3e3/lda.xlsx'
else:
    df = initial[initial['APOE'].isin([22, 23, 32, 33])] # APOE2
    datafolder = 'results/GNPC/e2vse3e3'
    outpath = 'results/GNPC/e2vse3e3/lda.xlsx'

In [18]:
f'LDA for {apoe_identifier} analysis, {len(df)} individuals'

'LDA for APOE4 analysis, 2864 individuals'

In [19]:
Counter(df['APOE'])

Counter({33: 1679, 34: 1013, 44: 172})

# Processing

In [20]:
protein_cols = [x for x in df.columns if x[:3] == 'seq']
covariates = ['age_at_visit', 'sex', 'plasma_ANML_mean', 'contributor_code']

residuals_df = pd.DataFrame(index=df.index)

In [21]:
for protein in protein_cols:
    tmp = df[covariates + [protein]]
    df_encoded = pd.get_dummies(df[['contributor_code']], drop_first=True).astype(int)
    X_all = pd.concat([tmp, df_encoded], axis=1)
    X_all = X_all.drop(columns='contributor_code')
    
    covariates_new = [x for x in X_all.columns if x[:3] != 'seq']
    hasvalue = X_all[~X_all[protein].isna()]
    novalue = X_all[X_all[protein].isna()]
    
    X = hasvalue[covariates_new]
    y = hasvalue[[protein]]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    
    hasvalue[protein] = model.resid
    novalue[protein] = np.nan
    
    merged = pd.concat([hasvalue, novalue], join='inner')

    residuals_df = pd.merge(residuals_df, merged[[protein]], left_index=True, right_index=True, how='left')

In [23]:
len(residuals_df)

2864

# Load residuals

In [ ]:
Results = ResultDataLoader('results/GNPC/e4vse3e3' if apoe_identifier=='APOE4' else 'results/GNPC/e2vse3e3', 
                           'GNPC-plasma-SomaLogic', apoe_identifier, 'AD','ref_name', 'GNPC')


# Train and fit for all

In [ ]:
apoe_specific = list(Results.apoe_solo_final)
ad_specific = list(Results.pathology_solo)
path1 = list(Results.path1_proteins)
path2 = list(Results.path2_proteins)
other = list(Results.apoe_result.sig - Results.apoe_solo_final - Results.path1_proteins - Results.path2_proteins)
rest = list(set(Results.apoe_result.data['Protein_id'].tolist()) - set(apoe_specific) - set(ad_specific) - set(path1) - set(path2) - set(other))

X_apoe4 = np.array(residuals_df[apoe_specific].T)
X_ab = np.array(residuals_df[ad_specific].T)
X_path1 = np.array(residuals_df[path1].T)
X_path2 = np.array(residuals_df[path2].T)
X_other = np.array(residuals_df[other].T)
X_rest = np.array(residuals_df[rest].T)

X_train = np.vstack([X_apoe4, X_ab, X_path1, X_path2, X_other])
y_train = np.array(
    [0] * len(X_apoe4) +
    [1] * len(X_ab) +
    [2] * len(X_path1) +
    [3] * len(X_path2) +
    [4] * len(X_other)
)

train_labels = [Results.id2symbol[x] for x in apoe_specific + ad_specific + path1 + path2 + other]
rest_labels = [Results.id2symbol[x] for x in rest]

lda = LDA()
lda.fit(X_train, y_train)

X_all = np.vstack([X_train, X_rest])
X_all_proj = lda.transform(X_all)

# Save for R plot

In [ ]:
y_pred_train = lda.predict(X_train)
#y_all = np.vstack([y_train, [5] * len(X_rest)])
print("Train accuracy:", accuracy_score(y_train, y_pred_train))
print("Classification Report:\n", classification_report(y_train, y_pred_train))

all_labels = train_labels + rest_labels

if apoe_identifier == 'APOE4':
    all_groups = (
        ['APOE4-specific'] * len(X_apoe4) +
        ['AD-specific'] * len(X_ab) +
        ['APOE4=>protein=>AD'] * len(X_path1) +
        ['APOE4=>AD=>protein'] * len(X_path2) +
        ['Non-specific'] * len(X_other) +
        ['Ungrouped'] * len(X_rest)
    )
else:
    all_groups = (
    ['APOE2-specific'] * len(X_apoe4) +
    ['AD-specific'] * len(X_ab) +
    ['APOE2=>protein=>AD'] * len(X_path1) +
    ['APOE2=>AD=>protein'] * len(X_path2) +
    ['Non-specific'] * len(X_other) +
    ['Ungrouped'] * len(X_rest)
    )
    
plot_df = pd.DataFrame({
    'Protein': all_labels,
    'Group': all_groups,
    'LD1': X_all_proj[:, 0],
    'LD2': X_all_proj[:, 1]
})

plot_df.to_excel(outpath, index=False)

Train accuracy: 0.9975609756097561
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99        72
           1       1.00      1.00      1.00      3333
           2       1.00      1.00      1.00        16
           3       0.99      0.98      0.99       200
           4       1.00      0.96      0.98        69

    accuracy                           1.00      3690
   macro avg       1.00      0.98      0.99      3690
weighted avg       1.00      1.00      1.00      3690



In [29]:
plot_df[plot_df['Protein']=='SPC25']

,Protein,Group,LD1,LD2
3413,SPC25,APOE4=>protein=>AD,-11.157497,3.341695
